# Módulo 4: Aprendizaje Automático (Machine Learning)
## Semana 2 - Clase 6b: Optimización, Validación Cruzada y Métricas Avanzadas

La presente sesión tiene como objetivo dotar al estudiante de técnicas formales para la evaluación y optimización de modelos predictivos. Se superará el enfoque tradicional de división simple (entrenamiento/prueba) introduciendo la Validación Cruzada (k-Fold), y se automatizará el hallazgo de hiperparámetros óptimos mediante `GridSearchCV`.

---
## **Sección 1: Extracción y Documentación de Datos Médicos**

Se empleará la base de datos de Cáncer de Mama (Wisconsin). Para facilitar la interpretación clínica de los resultados, se ha procedido a traducir y documentar las variables extraídas de las biopsias celulares.

### **Mapeo de Variables y Correlación Clínica Esperada**

Las células cancerosas malignas suelen exhibir un comportamiento caótico de crecimiento, perdiendo su estructura circular natural. Por lo tanto, el sistema de aprendizaje automático buscará correlacionar alteraciones físicas extremas con la malignidad.

*   **Radio, Perímetro y Área (Radius, Perimeter, Area):** Dimensiones del núcleo celular. *Correlación:* Valores excesivamente grandes o anormalmente pequeños (varianza alta) están directamente vinculados a tumores malignos, dado que las células cancerosas crecen sin control.
*   **Suavidad (Smoothness):** Variación en la longitud de los radios. *Correlación:* Bordes muy irregulares o dentados indican una alta probabilidad de malignidad.
*   **Concavidad y Puntos Cóncavos (Concavity, Concave points):** Severidad de las hendiduras en la célula. *Correlación:* Las células sanas tienden a ser esféricas (convexas). Alta concavidad es un indicador fuerte de deformación maligna.
*   **Simetría (Symmetry):** *Correlación:* Células asimétricas suelen ser un síntoma de división celular defectuosa (cáncer).

A continuación, se procederá a traducir programáticamente las características al idioma español.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer

# Extracción de datos
datos_cancer = load_breast_cancer()
X = pd.DataFrame(datos_cancer.data, columns=datos_cancer.feature_names)

# Traducción masiva de la nomenclatura médica al español
X.columns = [
    col.replace('mean', 'media')
       .replace('worst', 'peor')
       .replace('error', 'error')
       .replace('radius', 'radio')
       .replace('texture', 'textura')
       .replace('perimeter', 'perimetro')
       .replace('area', 'area')
       .replace('smoothness', 'suavidad')
       .replace('compactness', 'compacidad')
       .replace('concave points', 'puntos_concavos')
       .replace('concavity', 'concavidad')
       .replace('symmetry', 'simetria')
       .replace('fractal dimension', 'dimension_fractal')
       .replace(' ', '_')
    for col in X.columns
]

# Definición del vector objetivo: 1 = Maligno (Malignant), 0 = Benigno (Benign)
y = pd.Series(datos_cancer.target).map({0: 1, 1: 0}) 

display(X.head())

---
## **Sección 2: El Dilema Sesgo-Varianza y Validación Cruzada**

### **Concepto Teórico**
*   **Alto Sesgo (Underfitting):** El modelo asume relaciones demasiado simples y es incapaz de capturar la complejidad de los datos.
*   **Alta Varianza (Overfitting):** El modelo memoriza el ruido específico de los datos de entrenamiento, perdiendo la capacidad de generalizar a pacientes nuevos.

Para estimar el rendimiento real de un modelo sin el sesgo introducido por un solo corte aleatorio (`train_test_split`), se emplea la **Validación Cruzada k-Fold**. Este método divide el dataset en *k* fragmentos iguales, entrenando el modelo *k* veces y utilizando un fragmento distinto como prueba en cada iteración.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score

# Se instancia un modelo base (Árbol de Decisión sin límites)
modelo_base = DecisionTreeClassifier(random_state=42)

# Ejecución de Validación Cruzada con 5 particiones (k=5)
resultados_cv = cross_val_score(modelo_base, X, y, cv=5, scoring='accuracy')

print("Rendimiento en cada partición individual:")
print(np.round(resultados_cv, 4))
print(f"\nExactitud Promedio (Validación Cruzada): {resultados_cv.mean() * 100:.2f}%")
print(f"Desviación Estándar (Varianza del modelo): ±{resultados_cv.std() * 100:.2f}%")

---
## **Sección 3: Métricas Avanzadas (Más allá de la Exactitud)**

En contextos médicos, la Exactitud global es engañosa. Se hace necesario incorporar:
*   **Sensibilidad o Recall:** Proporción de tumores malignos reales que el sistema logró detectar (Minimización de Falsos Negativos).
*   **Precisión:** Proporción de predicciones malignas que fueron correctas.
*   **F1-Score:** Media armónica entre Sensibilidad y Precisión.

Además, se utilizará la curva **ROC (Receiver Operating Characteristic)** y la métrica **AUC (Area Under Curve)**. Un AUC de 1.0 indica un modelo perfecto, mientras que un AUC de 0.5 representa una predicción aleatoria.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

# Se realiza una partición estándar para la demostración gráfica
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

modelo_base.fit(X_train, y_train)
predicciones = modelo_base.predict(X_test)

print("Reporte de Clasificación Médico:\n")
print(classification_report(y_test, predicciones))

# Cálculo de probabilidades para la Curva ROC (Se requieren las probabilidades, no solo la decisión 0 o 1)
probabilidades_maligno = modelo_base.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, probabilidades_maligno)

# Construcción gráfica de la Curva ROC
fpr, tpr, umbrales = roc_curve(y_test, probabilidades_maligno)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'Curva ROC (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Adivinación Aleatoria (AUC = 0.5)')
plt.xlabel('Tasa de Falsos Positivos (1 - Especificidad)')
plt.ylabel('Tasa de Verdaderos Positivos (Sensibilidad/Recall)')
plt.title('Curva ROC - Rendimiento de Clasificación')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

---
## **Sección 4: Automatización con Pipelines y GridSearchCV**

Para encontrar los parámetros óptimos (hiperparámetros) que maximicen el AUC, se construirá un ecosistema automatizado. Se encapsulará el escalado de características y el algoritmo predictivo dentro de un objeto `Pipeline`.

Posteriormente, `GridSearchCV` someterá este Pipeline a múltiples pruebas exhaustivas de Validación Cruzada modificando sistemáticamente los hiperparámetros.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV

# 1. Ensamblaje del Pipeline secuencial
pipeline_knn = Pipeline(steps=[
    ('escalador', StandardScaler()),
    ('clasificador', KNeighborsClassifier())
])

# 2. Definición de la cuadrícula de parámetros a explorar.
# Note la sintaxis: nombre_del_paso__parametro
grilla_parametros = {
    'clasificador__n_neighbors': [3, 5, 7, 9, 11],
    'clasificador__weights': ['uniform', 'distance'],
    'clasificador__p': [1, 2] # 1=Manhattan, 2=Euclidiana
}

# 3. Configuración de GridSearchCV priorizando la métrica AUC (roc_auc)
optimizador = GridSearchCV(
    estimator=pipeline_knn, 
    param_grid=grilla_parametros, 
    cv=5,                 # Validación Cruzada de 5 pliegues
    scoring='roc_auc',    # Métrica de éxito exigida
    n_jobs=-1             # Uso de todos los procesadores disponibles
)

# 4. Ejecución masiva del entrenamiento y optimización
print("Iniciando búsqueda exhaustiva de hiperparámetros...")
optimizador.fit(X_train, y_train)

print("\n--- Resultados de la Optimización ---")
print(f"Mejor AUC durante Cross-Validation: {optimizador.best_score_:.4f}")
print("Mejores Parámetros encontrados:")
for clave, valor in optimizador.best_params_.items():
    print(f" - {clave}: {valor}")

Finalmente, se evalúa el modelo óptimo contra el conjunto de pruebas que permaneció oculto (Hold-out test set).

In [ ]:
# El objeto 'optimizador' automáticamente retiene el mejor modelo entrenado.
mejor_modelo = optimizador.best_estimator_

# Predicción de probabilidades para AUC en datos no vistos
proba_test = mejor_modelo.predict_proba(X_test)[:, 1]
auc_final = roc_auc_score(y_test, proba_test)

print(f"Área Bajo la Curva (AUC) Final sobre el Set de Prueba: {auc_final:.4f}")